In [1]:
from pathlib import Path

# 找 project root（更稳）
BASE_DIR = Path.cwd()

def find_project_root(base_dir: Path) -> Path:
    for parent in [base_dir] + list(base_dir.parents):
        if (parent / "data").exists():
            return parent
    raise FileNotFoundError("❌ Cannot find project root (no 'data' folder found)")

project_root = find_project_root(BASE_DIR)

# 数据目录（你现在用 txt，不是 md）
data_dir = project_root / "data" / "processed" / "pdf_parsed"/"CAR-T Products Brochure.txt"

In [7]:
import re
import pandas as pd

# 读取txt文件
with open(data_dir, "r", encoding="utf-8") as f:
    text = f.read()

# 找到所有 PRODUCT 块
products = re.findall(r"\[PRODUCT\](.*?)\[END_PRODUCT\]", text, re.DOTALL)

data = []

for p in products:
    item = {}
    
    # 提取字段
    catalog = re.search(r"catalog_no:\s*(.*)", p)
    target = re.search(r"target_antigen:\s*(.*)", p)
    domain = re.search(r"costimulatory_domain:\s*(.*)", p)
    construct = re.search(r"construct:\s*(.*)", p)
    price = re.search(r"price_usd:\s*(.*)", p)

    item["catalog_no"] = catalog.group(1) if catalog else None
    item["target_antigen"] = target.group(1) if target else None
    item["costimulatory_domain"] = domain.group(1) if domain else None
    item["construct"] = construct.group(1) if construct else None
    item["price_usd"] = price.group(1) if price else None

    data.append(item)

# 转成DataFrame
df = pd.DataFrame(data)

# 导出Excel
df.to_excel(project_root / "data" / "processed" / "CAR-T_products.xlsx", index=False)

print("Done! 已生成 Excel 文件")

Done! 已生成 Excel 文件


In [8]:
import re
import pandas as pd
from pathlib import Path

# ========= 1. 找到 project root =========
BASE_DIR = Path.cwd()

if (BASE_DIR / "data").exists():
    project_root = BASE_DIR
else:
    project_root = BASE_DIR.parent

# ========= 2. 指定 txt 文件 =========
txt_path = project_root / "data" / "processed" / "pdf_parsed" / "mRNA-LNP Products & Services Brochure.txt"

# ========= 3. 读取文本 =========
with open(txt_path, "r", encoding="utf-8") as f:
    text = f.read()

# ========= 4. 提取所有 [PRODUCT] 块 =========
product_blocks = re.findall(r"\[PRODUCT\](.*?)\[END_PRODUCT\]", text, re.DOTALL)

# ========= 5. 逐个解析 =========
data = []

for block in product_blocks:
    row = {}

    name = re.search(r"name:\s*(.*)", block)
    catalog_no = re.search(r"catalog_no:\s*(.*)", block)
    product_type = re.search(r"type:\s*(.*)", block)
    price_usd = re.search(r"price_usd:\s*(.*)", block)
    product_format = re.search(r"format:\s*(.*)", block)

    row["name"] = name.group(1).strip() if name else None
    row["catalog_no"] = catalog_no.group(1).strip() if catalog_no else None
    row["type"] = product_type.group(1).strip() if product_type else None
    row["price_usd"] = price_usd.group(1).strip() if price_usd else None
    row["format"] = product_format.group(1).strip() if product_format else None

    data.append(row)

# ========= 6. 转成 DataFrame =========
df = pd.DataFrame(data)

# ========= 7. 导出 Excel =========
output_path = project_root / "data" / "processed" / "mRNA_LNP_products.xlsx"
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_excel(output_path, index=False)

print("✅ Saved to:", output_path)
print(df.head())

✅ Saved to: /Users/promab/anaconda_projects/email_agent/data/processed/mRNA_LNP_products.xlsx
                                              name   catalog_no     type  \
0  COVID-19 Spike Protein (Alpha Variant) mRNA-LNP  PM-LNP-0010  Protein   
1  COVID-19 Spike Protein (Alpha Variant) mRNA-LNP  PM-LNP-0011  Protein   
2  COVID-19 Spike Protein (Delta Variant) mRNA-LNP  PM-LNP-0012  Protein   
3           COVID-19 Nucleocapsid Protein mRNA-LNP  PM-LNP-0013  Protein   
4                            COVID-19 RBD mRNA-LNP  PM-LNP-0014  Protein   

  price_usd         format  
0       723  10ug in 200ul  
1       723  10ug in 200ul  
2       723  10ug in 200ul  
3       723  10ug in 200ul  
4       723  10ug in 200ul  
